In [19]:
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
    ParentDocumentRetriever
)

from langchain_classic.retrievers.document_compressors import (
    LLMListwiseRerank,
    LLMChainExtractor
)

from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [20]:
question = "What nutrients do apples provide?"

In [21]:
from langchain_community.document_loaders import TextLoader
# Document Loading:
loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")

documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")

documents_diet = loader_diet.load()

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#  Text Splitting:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=1
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

#Embedding Generation, Vector Storage
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)
vectorstore_fruits = FAISS.from_documents(chunks_fruits, embedding_model)
vectorstore_fruits.save_local("./faiss_fruits")

vectorstore_diet = FAISS.from_documents(chunks_diet, embedding_model)
vectorstore_diet.save_local("./faiss_diet")

In [24]:

# --------------------------------------------------
# Prepare combined fruits and nutrition documents
# --------------------------------------------------

for doc in chunks_fruits:
    doc.metadata["topic"] = "fruits"

for doc in chunks_diet:
    doc.metadata["topic"] = "nutrition"

all_chunks = chunks_fruits + chunks_diet

# Build a combined FAISS database
combined_vectorstore = FAISS.from_documents(
    all_chunks, embedding_model
)

In [25]:
"""
BM25 Retriever is a keyword-based search mechanism that finds documents by checking how closely their actual words match the search query. Unlike FAISS, which uses embeddings to find semantic similarity (meaning), BM25 focuses on word occurrence and relevance, giving more importance to meaningful terms that appear frequently in a document but are less common across the overall collection. For example, searching for "vitamins in apples" can retrieve documents containing those exact terms. In Hybrid Search, we combine BM25 with FAISS so that we benefit from both keyword matching and semantic similarity.
"""

'\nBM25 Retriever is a keyword-based search mechanism that finds documents by checking how closely their actual words match the search query. Unlike FAISS, which uses embeddings to find semantic similarity (meaning), BM25 focuses on word occurrence and relevance, giving more importance to meaningful terms that appear frequently in a document but are less common across the overall collection. For example, searching for "vitamins in apples" can retrieve documents containing those exact terms. In Hybrid Search, we combine BM25 with FAISS so that we benefit from both keyword matching and semantic similarity.\n'

In [26]:
# You're thinking correctly! We have one collection of documents (all_chunks) but two different search engines. BM25 takes those chunks as plain text and builds its own keyword-search index, so it doesn't need an embedding model. FAISS takes the same chunks, converts them into numerical vectors using embedding_model, and searches by semantic similarity. Then EnsembleRetriever acts like an assembly line coordinator: it sends the same question to both retrievers, combines their results and uses the weights [0.5, 0.5] to give equal importance to both search methods. So we're not creating another vectorstore; we're combining two independent retrieval mechanisms to get better search results.

In [ ]:

# --------------------------------------------------
# 1. HYBRID SEARCH ---LLM PRESENT 
# Combine keyword search (BM25) and vector search (FAISS)
# --------------------------------------------------

bm25_retriever = BM25Retriever.from_documents(all_chunks)
bm25_retriever.k = 4

vector_retriever = combined_vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

hybrid_docs = hybrid_retriever.invoke(question)

In [28]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.2)

In [ ]:
# LLMListwiseRerank uses an LLM to examine a group of retrieved documents together and reorder them according to how relevant they are to the user's question. For example, Hybrid Search might return 8 documents; the reranker asks your LLM to compare their relevance and select the top 2. Unlike FAISS, which ranks using embedding similarity, reranking uses the LLM's understanding of the actual question and document content. In our setup, Qwen performs this reranking through Ollama.

In [ ]:

# --------------------------------------------------
# 2. RERANKING LLM PRESENT
# : Reranking selects the best books;
# Reorder retrieved documents by relevance to the question
# Reranking is like sorting the retrieved documents again, but this time using an LLM's understanding of relevance rather than just the original search scores.
# --------------------------------------------------

from langchain_ollama import ChatOllama

llm_structured = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)
reranker = LLMListwiseRerank.from_llm(
    llm_structured, top_n=2
)

reranked_docs = reranker.compress_documents(
    documents=hybrid_docs,
    query=question
)

In [ ]:
# Reranking is about choosing the most relevant documents and putting them in the right order, whereas Contextual Compression is about removing irrelevant portions from those documents while retaining useful information. We compress after reranking because we've already identified the best documents; now we want to extract only the information needed to answer the question, reducing unnecessary context sent to the final LLM. Think of it this way: Reranking selects the best books; Contextual Compression highlights only the relevant paragraphs inside those books.

In [ ]:
# --------------------------------------------------
# 3. CONTEXTUAL COMPRESSION LLM PRESENT
# Extract only the relevant information from documents
# Contextual Compression highlights only the relevant paragraphs inside those books.
# removes irrelevant documents, takes more time
# --------------------------------------------------

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_retriever,
    base_compressor=compressor
)

compressed_docs = compression_retriever.invoke(question)

In [34]:
"""
Reranking When FAISS returns top-k chunks, it ranks them purely by vector similarity — which is fast but not always accurate. Reranking is a second pass where a more powerful cross-encoder model looks at the query and each retrieved chunk together and rescores them. Unlike embeddings which encode query and chunk independently, a cross-encoder reads both simultaneously and gives a much more precise relevance score. Think of FAISS retrieval as a rough shortlist and reranking as the final interview round.

Compression Sometimes your retrieved chunk is 500 tokens but only 2 sentences are actually relevant to the question. Compression means passing the retrieved chunk through an LLM and asking it to extract only the relevant parts before stuffing it into the final prompt. This saves tokens, reduces noise, and gives Qwen a cleaner context to reason from. LangChain calls this ContextualCompressionRetriever.

Metadata Filtering  Every chunk in your vectorstore has metadata — filename, page number, date, category, author etc. Metadata filtering lets you narrow the search space before doing similarity search — for example “only search chunks from documents uploaded this month” or “only search within the HR policy document”. This is why Chroma and Qdrant have an edge over raw FAISS — they support rich metadata filtering natively, making retrieval faster and more precise in production systems.


"""

'\nReranking When FAISS returns top-k chunks, it ranks them purely by vector similarity — which is fast but not always accurate. Reranking is a second pass where a more powerful cross-encoder model looks at the query and each retrieved chunk together and rescores them. Unlike embeddings which encode query and chunk independently, a cross-encoder reads both simultaneously and gives a much more precise relevance score. Think of FAISS retrieval as a rough shortlist and reranking as the final interview round.\n\nCompression Sometimes your retrieved chunk is 500 tokens but only 2 sentences are actually relevant to the question. Compression means passing the retrieved chunk through an LLM and asking it to extract only the relevant parts before stuffing it into the final prompt. This saves tokens, reduces noise, and gives Qwen a cleaner context to reason from. LangChain calls this ContextualCompressionRetriever.\n\nMetadata Filtering  Every chunk in your vectorstore has metadata — filename, p

In [ ]:

# --------------------------------------------------
# 4. METADATA FILTERING LLM ABSENT
# Retrieve documents belonging only to the fruits category
# --------------------------------------------------

fruits_filtered_retriever = combined_vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {"topic": "fruits"}
    }
)

fruits_filtered_docs = fruits_filtered_retriever.invoke(question)


In [36]:
"""
Parent Document Retrieval — This solves a fundamental tension in chunking: you want small chunks for accurate embedding (small chunks have focused meaning, so similarity search is precise) but large chunks for rich LLM context (the LLM needs surrounding context to answer well). Parent Document Retrieval solves this by maintaining two levels — it splits documents into small child chunks for embedding and similarity search, but when a child chunk is retrieved it hands back its larger parent chunk to the LLM. So you get the precision of small chunk retrieval with the context richness of large chunk generation. Think of it as — search using the index card, but read the full page.
"""

'\nParent Document Retrieval — This solves a fundamental tension in chunking: you want small chunks for accurate embedding (small chunks have focused meaning, so similarity search is precise) but large chunks for rich LLM context (the LLM needs surrounding context to answer well). Parent Document Retrieval solves this by maintaining two levels — it splits documents into small child chunks for embedding and similarity search, but when a child chunk is retrieved it hands back its larger parent chunk to the LLM. So you get the precision of small chunk retrieval with the context richness of large chunk generation. Think of it as — search using the index card, but read the full page.\n'

In [ ]:

# --------------------------------------------------
# 5. PARENT DOCUMENT RETRIEVAL LLM present
# Retrieve smaller child chunks but return their larger parent context
# InMemoryStore is just a Python dictionary under the hood — it stores the parent chunks in RAM as key-value pairs where the key is a chunk ID and the value is the actual parent document text.
# --------------------------------------------------

parent_docs = documents_fruits + documents_diet

for doc in documents_fruits:
    doc.metadata["topic"] = "fruits"

for doc in documents_diet:
    doc.metadata["topic"] = "nutrition"

parent_retriever = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embedding=embedding_model),
    docstore=InMemoryStore(),
    parent_splitter=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    ),
    child_splitter=RecursiveCharacterTextSplitter(
        chunk_size=250,
        chunk_overlap=30
    )
)

parent_retriever.add_documents(parent_docs)

parent_results = parent_retriever.invoke(question)


In [42]:
# --------------------------------------------------
# Display results
# --------------------------------------------------

print("\n--- HYBRID SEARCH ---")
for doc in hybrid_docs:
    print(doc.page_content)

print("\n--- RERANKING ---")
for doc in reranked_docs:
    print(doc.page_content)

# print("\n--- CONTEXTUAL COMPRESSION ---")
# for doc in compressed_docs:
#     print(doc.page_content)

print("\n--- METADATA FILTERING ---")
for doc in fruits_filtered_docs:
    print(doc.page_content)

print("\n--- PARENT DOCUMENT RETRIEVAL ---")
for doc in parent_results:
    print(doc.page_content)


context = doc.page_content


--- HYBRID SEARCH ---
A balanced diet provides the nutrients and energy required by the body without excessive or insufficient intake. It generally includes vegetables, fruits, whole grains, protein-rich foods and appropriate amounts of healthy fats.

Eating different coloured fruits and vegetables helps provide a variety of vitamins, minerals and plant compounds. Whole grains provide carbohydrates and fibre, while protein-rich foods support tissue growth and repair.
# HUMAN NUTRITION AND DIETARY REQUIREMENTS

## Introduction

Human nutrition is the process through which the body obtains and uses nutrients from food. These nutrients provide energy, support growth, repair body tissues and help maintain normal body functions. A balanced diet contains different types of nutrients in appropriate quantities. Nutritional requirements vary depending on age, physical activity, body size, health conditions and lifestyle.

## 1. Macronutrients
Fruits and vegetables both come from plants. They c

In [ ]:
# Both connect LangChain to your local Ollama model, but OllamaLLM is designed for plain text completion, where you send text and receive text, whereas ChatOllama is designed for conversational interactions, using structured messages such as System, Human and AI messages. ChatOllama also supports chat-specific capabilities such as tool calling and structured output (depending on model and configuration), which is why it is more suitable for components like LLMListwiseRerank. Think of OllamaLLM as a basic text interface and ChatOllama as a richer conversational interface.

In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.2)
# Prepare the final prompt with retrieved context and original question
final_prompt = ChatPromptTemplate.from_template("""
Answer the question using the following context.

Context:
{context}

Question:
{question}

Answer:
""")

# Pass the context and question to the LLM
final_chain = final_prompt | llm | StrOutputParser()

# Generate the final answer
response = final_chain.invoke({
    "context": context,
    "question": question
})

print("Final Answer:", response)

Final Answer: The context provided does not mention apples or any specific nutrients that apples provide. The information given is about proteins, fats, and micronutrients, but does not include any information about apples or their nutritional content.
